# LV2 - Uvod u Python (2/2)

## Zadatak 1 - Obrada podataka kroz Pandas

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# a) Ucitaj CSV i prikazi prvih 5 i posljednjih 10 unosa
data = pd.read_csv('izvjestaj.csv')
print('Prvih 5:')
display(data.head(5))
print('Posljednjih 10:')
display(data.tail(10))

In [ ]:
# b) Ispisati samo podatke za redovne parcijalne ispite (Ispit1 i Ispit2)
display(data[['Ispit1', 'Ispit2']])

In [ ]:
# c) Ispisati sve studente koji su izgubili prisustvo
display(data.loc[data['Prisustvo'] == 0])

In [ ]:
# d) Indeks, UKUPNO i Ocjena za studente koji su upisali ocjenu 8 ili vise
# Ocjena kolona je tipa object zbog '/', pa moramo biti pazljivi
# Konvertujemo Ocjena u numeric, '/' postaje NaN
ocjena_num = pd.to_numeric(data['Ocjena'], errors='coerce')
display(data.loc[ocjena_num >= 8, ['Indeks', 'UKUPNO', 'Ocjena']])

In [ ]:
# e) Sve nedostajuce vrijednosti (oznacene sa /) zamijeniti sa np.nan
data.replace('/', np.nan, inplace=True)
display(data.head())

In [ ]:
# f) Odbaciti sve studente koji nemaju upisanu ocjenu
data.dropna(subset=['Ocjena'], inplace=True)
data = data.reset_index(drop=True)
display(data)

In [ ]:
# Konvertovati numericke kolone iz object u float (jer su imale '/' pa su object)
numeric_cols = ['Ispit1', 'Ispit2', 'Ispit1_popravni', 'Ispit2_popravni', 'UKUPNO', 'Ocjena', 'Prisustvo']
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

In [ ]:
# g) Kreirati novu kolonu Ispit1_final - uzima bolji rezultat od Ispit1 i Ispit1_popravni
# Ako student nije pisao popravni (NaN), uzima se redovni, i obrnuto
data['Ispit1_final'] = data[['Ispit1', 'Ispit1_popravni']].max(axis=1)
display(data[['Indeks', 'Ispit1', 'Ispit1_popravni', 'Ispit1_final']].head(10))

In [ ]:
# h) Ponoviti za Ispit2
data['Ispit2_final'] = data[['Ispit2', 'Ispit2_popravni']].max(axis=1)
display(data[['Indeks', 'Ispit2', 'Ispit2_popravni', 'Ispit2_final']].head(10))

In [ ]:
# i) Odbaciti cetiri stare kolone za ispite
data.drop(columns=['Ispit1', 'Ispit2', 'Ispit1_popravni', 'Ispit2_popravni'], inplace=True)
display(data.head())

In [ ]:
# j) Sacuvati kao CSV sa separatorom tacka-zarez
data.to_csv('izvjestaj_modificirano.csv', sep=';', index=False)
print('Sacuvano: izvjestaj_modificirano.csv')

In [ ]:
# k) Sacuvati kao Pickle datoteku
import pickle
pickle.dump(data, open('izvjestaj_modificirano_pickle.p', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)
print('Sacuvano: izvjestaj_modificirano_pickle.p')

---
## Zadatak 2 - Normalizacija podataka pomocu Sklearn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
# a) Ucitaj izvjestaj.csv
data2 = pd.read_csv('izvjestaj.csv')
data2.replace('/', np.nan, inplace=True)
numeric_cols2 = ['Ispit1', 'Ispit2', 'Ispit1_popravni', 'Ispit2_popravni', 'UKUPNO', 'Ocjena', 'Prisustvo']
for col in numeric_cols2:
    data2[col] = pd.to_numeric(data2[col], errors='coerce')
display(data2.head())

In [ ]:
# b) Redovni ispiti (Ispit1, Ispit2) - zamjena NaN sa median
si_median = SimpleImputer(strategy='median')
data2[['Ispit1', 'Ispit2']] = si_median.fit_transform(data2[['Ispit1', 'Ispit2']])
display(data2[['Ispit1', 'Ispit2']].head())

In [ ]:
# c) Popravni ispiti (Ispit1_popravni, Ispit2_popravni) - zamjena NaN sa mean
si_mean = SimpleImputer(strategy='mean')
data2[['Ispit1_popravni', 'Ispit2_popravni']] = si_mean.fit_transform(data2[['Ispit1_popravni', 'Ispit2_popravni']])
display(data2[['Ispit1_popravni', 'Ispit2_popravni']].head())

In [ ]:
# d) Z-score normalizacija za redovne ispite
data2['Ispit1'] = scale(data2['Ispit1'])
data2['Ispit2'] = scale(data2['Ispit2'])
display(data2[['Ispit1', 'Ispit2']].head())

In [ ]:
# e) MinMax normalizacija za popravne ispite
mm = MinMaxScaler()
data2[['Ispit1_popravni', 'Ispit2_popravni']] = mm.fit_transform(data2[['Ispit1_popravni', 'Ispit2_popravni']])
display(data2[['Ispit1_popravni', 'Ispit2_popravni']].head())

In [ ]:
# f) Sve ostale NaN zamijeniti sa nulama
data2.fillna(0, inplace=True)
display(data2.head())

In [ ]:
# g) Izdvojiti kolonu Ocjena u posebnu varijablu i ukloniti je iz DataFrame-a
ocjena = data2['Ocjena'].copy()
data2.drop(columns=['Ocjena'], inplace=True)
print('Ocjena kolona:')
print(ocjena)
print('\nDataFrame bez Ocjena kolone:')
display(data2.head())

In [ ]:
# h) Konvertovati u NumPy nizove
X = data2.to_numpy()
y = ocjena.to_numpy()
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# i) Podjela na trening i testni skup (20% testni)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Trening skup - X:', X_train.shape, '  y:', y_train.shape)
print('Testni skup  - X:', X_test.shape, '  y:', y_test.shape)

---
## Zadatak 3 - Klasicni algoritam masinskog ucenja (k-NN na Iris skupu)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

In [ ]:
# a) Ucitavanje Iris skupa podataka
iris = load_iris()
X = iris.data
y = iris.target

feature_names = iris.feature_names
target_names = iris.target_names

print('Nazivi znacajki:', feature_names)
print('Nazivi labela:', target_names)
print('\nPrvih 5 redova X:\n', X[:5])

In [ ]:
# b) Podjela skupa podataka - 70% trening, 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

In [ ]:
# c) Treniranje k-NN klasifikatora sa k=3
classifier_knn = KNeighborsClassifier(n_neighbors=3)
classifier_knn.fit(X_train, y_train)

In [ ]:
# d) Predikcija i racunanje tacnosti
y_pred = classifier_knn.predict(X_test)
print('Tacnost:', metrics.accuracy_score(y_test, y_pred))

In [ ]:
# e) Isprobati razlicite vrijednosti k
for k in [1, 3, 5, 7, 10, 15, 20, 45]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, pred)
    print(f'k={k:3d}  ->  tacnost: {acc:.4f}')

---
## Zadatak 4 - Duboko ucenje (MNIST klasifikacija cifara)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.datasets import mnist
from keras import models, layers
from keras.utils import to_categorical

In [ ]:
# a) Ucitavanje MNIST skupa podataka
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print('Trening skup - slike:', x_train.shape, '  labele:', y_train.shape)
print('Testni skup  - slike:', x_test.shape, '  labele:', y_test.shape)
print('Memorija (x_train):', x_train.nbytes / 1024, 'kB')
print('Memorija (x_test):', x_test.nbytes / 1024, 'kB')

In [ ]:
# b) Reshape i normalizacija (svodi piksele sa [0,255] na [0,1])
train_images = x_train.reshape((x_train.shape[0], 28 * 28))
train_images = train_images.astype('float32') / 255

test_images = x_test.reshape((x_test.shape[0], 28 * 28))
test_images = test_images.astype('float32') / 255

print('train_images shape:', train_images.shape, '  memorija:', train_images.nbytes / 1024, 'kB')
print('test_images shape:', test_images.shape, '  memorija:', test_images.nbytes / 1024, 'kB')

In [ ]:
# c) Kreiranje modela neuralne mreze
network = models.Sequential()
network.add(layers.Dense(512, activation='relu', input_shape=(28 * 28,)))
network.add(layers.Dense(10, activation='softmax'))
network.summary()

In [ ]:
# d) Kompajliranje modela
network.compile(optimizer='rmsprop',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

In [ ]:
# e) One-hot kodiranje labela
train_labels = to_categorical(y_train)
test_labels = to_categorical(y_test)
print('train_labels shape:', train_labels.shape)
print('Primjer one-hot kodiranja za cifru', y_train[0], ':', train_labels[0])

In [ ]:
# f) Treniranje mreze
network.fit(train_images, train_labels, epochs=5, batch_size=128)

In [ ]:
# g) Evaluacija nad testnim skupom
test_loss, test_acc = network.evaluate(test_images, test_labels)
print('Tacnost za testni skup:', test_acc)